# imports

In [ ]:
import os
import random
import torchvision
import numpy as np
from PIL import Image
import cv2
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from skimage.feature import local_binary_pattern
from skimage.filters.rank import entropy
from skimage.morphology import disk
import torchvision.transforms as transforms

In [ ]:
!git pull

# sync github

In [ ]:
repo_name = "SIV-Texture-Anomaly-detection"
git_url = "https://github.com/Marco1743/SIV-Texture-Anomaly-detection.git"
if not os.path.exists(repo_name):
    !git clone {git_url}
%cd {repo_name}
!git pull

# import dataset

In [ ]:
os.makedirs('data/train/good', exist_ok=True)
os.makedirs('data/test/anomaly', exist_ok=True)

print("Scaricamento Texture reali (DTD Dataset)...")
dtd_train = torchvision.datasets.DTD(root='./data', split='train', download=True)
dtd_test = torchvision.datasets.DTD(root='./data', split='test', download=True)

## creazione train e test

In [ ]:
print("creazione training")
count = 0
for img, label in dtd_train:
    img_gray = img.convert('L')
    img_gray.save(f'data/train/good/tex_{count}.png')
    count += 1

print("Generazione anomalie...")
for i in range(50):
    img, _ = dtd_test[i]
    img_arr = np.array(img.convert('L'))
    h, w = img_arr.shape

    tipo_anomalia = random.choice([0, 1])

    if tipo_anomalia == 0:
        box_h = random.randint(10, 50)
        box_w = random.randint(10, 50)
        y = random.randint(0, h - box_h)
        x = random.randint(0, w - box_w)
        intensita = random.randint(0, 60)

        img_arr[y:y+box_h, x:x+box_w] = intensita

    else:
        spessore = random.randint(2, 6)
        lunghezza = random.randint(40, 100)

        if random.choice([True, False]):
            y = random.randint(0, h - spessore)
            x = random.randint(0, w - lunghezza)
            img_arr[y:y+spessore, x:x+lunghezza] = 255
        else:
            y = random.randint(0, h - lunghezza)
            x = random.randint(0, w - spessore)
            img_arr[y:y+lunghezza, x:x+spessore] = 255

    Image.fromarray(img_arr).save(f'data/test/anomaly/anom_{i}.png')

print(f"Dataset pronto e processato con augmentations.py! Totale: {count}")

# PIPELINE 1

## Carichiamo l'immagine

In [ ]:
img_path = 'data/test/anomaly/anom_0.png'
img = cv2.imread(img_path, 0)

## Estrazione LBP, Entropia Locale sull'LBP and Segmentazione K-Means sulla mappa di Entropia

In [ ]:
import cv2
import numpy as np
from skimage.feature import local_binary_pattern
from skimage.filters.rank import entropy
from skimage.morphology import disk
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

if hasattr(img, 'numpy'):
    img = img.squeeze().cpu().numpy()
    if img.ndim == 3 and img.shape[0] in [1, 3]:
        img = np.transpose(img, (1, 2, 0))

if not isinstance(img, np.ndarray):
    img = np.array(img)

if len(img.shape) == 3:
    img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

if img.dtype != np.uint8:
    if img.max() <= 1.0:
        img = (img * 255).astype(np.uint8)
    else:
        img = img.astype(np.uint8)

img_blur = cv2.GaussianBlur(img, (5, 5), 0)

radius = 3
n_points = 8 * radius
lbp_img = local_binary_pattern(img_blur, n_points, radius, method='uniform')
lbp_uint8 = cv2.normalize(lbp_img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

entropia_locale = entropy(lbp_uint8, disk(15))

X_features = np.column_stack((entropia_locale.flatten(), img_blur.flatten()))
X_scaled = StandardScaler().fit_transform(X_features)

kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
kmeans.fit(X_scaled)
anomaly_mask = kmeans.labels_.reshape(img.shape).astype(np.uint8)

if np.sum(anomaly_mask) > (anomaly_mask.size / 2):
    anomaly_mask = 1 - anomaly_mask

kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

anomaly_mask_clean = cv2.morphologyEx(anomaly_mask, cv2.MORPH_OPEN, kernel)

anomaly_mask_clean = cv2.morphologyEx(anomaly_mask_clean, cv2.MORPH_CLOSE, kernel)

ModuleNotFoundError: No module named 'cv2'

## Visualizzazione

In [ ]:
plt.figure(figsize=(20, 5))

plt.subplot(1, 4, 1)
plt.title("Immagine con Anomalia", fontsize=14)
plt.imshow(img, cmap='gray')
plt.axis('off')

plt.subplot(1, 4, 2)
plt.title("1. Feature LBP (Cruda)", fontsize=14)
plt.imshow(lbp_img, cmap='viridis')
plt.axis('off')

plt.subplot(1, 4, 3)
plt.title("2. Entropia Locale (Il Segreto!)", fontsize=14)
plt.imshow(entropia_locale, cmap='inferno')
plt.axis('off')

plt.subplot(1, 4, 4)
plt.title("3. Segmentazione K-Means", fontsize=14)
plt.imshow(anomaly_mask, cmap='gray')
plt.axis('off')

plt.tight_layout()
plt.show()

# PIPELINE 2

## dataloader

In [ ]:
from dataset import MoCoTextureDataset
print("Inizializzazione del PyTorch Dataset per MoCo...")
train_dataset = MoCoTextureDataset(data_dir='data/train/good', is_train=True)
view_1, view_2 = train_dataset[5]

to_pil = transforms.ToPILImage()
img_1 = to_pil(view_1)
img_2 = to_pil(view_2)

plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.title("Vista 1 (Augmentation Casuale A)", fontsize=14)
plt.imshow(img_1, cmap='gray')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title("Vista 2 (Augmentation Casuale B)", fontsize=14)
plt.imshow(img_2, cmap='gray')
plt.axis('off')

plt.tight_layout()
plt.show()

print(f"Shape del tensore 1: {view_1.shape}")
print(f"Shape del tensore 2: {view_2.shape}")

## test caricamento immagini

In [ ]:
!git pull

import torch
from torch.utils.data import DataLoader
from dataset import MoCoTextureDataset
from model import MoCo

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device in uso: {device}")

train_dataset = MoCoTextureDataset(data_dir='data/train/good', is_train=True)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
print("DataLoader pronto.")

model = MoCo(dim=128, K=1024).to(device)

view_1, view_2 = next(iter(train_loader))

view_1 = view_1.to(device)
view_2 = view_2.to(device)

logits, labels = model(view_1, view_2)

print(f"Input Shape: {view_1.shape}  --> [Batch Size, Canali, Altezza, Larghezza]")
print(f"Logits Shape: {logits.shape} --> [Batch Size, 1 (Risposta Giusta) + K (Risposte Sbagliate)]")
print(f"Labels Shape: {labels.shape}     --> [Batch Size]")

## pipeline moco

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from dataset import MoCoTextureDataset
from model import MoCo
import matplotlib.pyplot as plt
from tqdm import tqdm 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device in uso: {device}")

BATCH_SIZE = 32
EPOCHS = 50
LR = 0.001

train_dataset = MoCoTextureDataset(data_dir='data/train/good', is_train=True)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    num_workers=2,
    pin_memory=True
)

model = MoCo(dim=128, K=1024).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

print(f"\n INIZIO ADDESTRAMENTO ({EPOCHS} Epoche) 🚀")
losses = []

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    loop = tqdm(train_loader, desc=f"Epoca [{epoch+1}/{EPOCHS}]")

    for view_1, view_2 in loop:
        view_1, view_2 = view_1.to(device), view_2.to(device)

        optimizer.zero_grad()
        logits, labels = model(view_1, view_2)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(train_loader)
    losses.append(avg_loss)
    print(f"-> Fine Epoca {epoch+1} | Loss Media: {avg_loss:.4f}\n")

print("\nADDESTRAMENTO CONCLUSO")
torch.save(model.encoder_q.state_dict(), "moco_encoder_weights.pth")

plt.figure(figsize=(10, 5))
plt.plot(range(1, EPOCHS+1), losses, marker='o', linestyle='-', color='b')
plt.title("Curva di Apprendimento (Contrastive Loss)", fontsize=14)
plt.xlabel("Epoca", fontsize=12)
plt.ylabel("Loss InfoNCE", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

## test moco

In [ ]:
import torch
import torch.nn.functional as F
import torchvision.transforms as transforms
from PIL import Image
import os
import matplotlib.pyplot as plt
import numpy as np

from model import create_encoder

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder = create_encoder(dim_projection=128).to(device)
encoder.load_state_dict(torch.load("moco_encoder_weights.pth", map_location=device))
encoder.eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

train_dir = 'data/train/good'
train_paths = [os.path.join(train_dir, f) for f in os.listdir(train_dir)][:100]

normal_features = []
with torch.no_grad():
    for path in train_paths:
        img = Image.open(path).convert('L')
        img_t = transform(img).unsqueeze(0).to(device)
        feat = encoder(img_t)
        feat = F.normalize(feat, dim=1)
        normal_features.append(feat)

normal_bank = torch.cat(normal_features, dim=0)

def get_anomaly_score(img_path):
    img = Image.open(img_path).convert('L')
    img_t = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        feat = encoder(img_t)
        feat = F.normalize(feat, dim=1)

    similarities = torch.mm(feat, normal_bank.T) 

    max_sim = torch.max(similarities).item()

    anomaly_score = 1.0 - max_sim
    return img, anomaly_score

img_sana, score_sano = get_anomaly_score(train_paths[0])
img_anomala, score_anomalo = get_anomaly_score('data/test/anomaly/anom_0.png')

print(f"-> Score Immagine SANA:    {score_sano:.4f}")
print(f"-> Score Immagine ANOMALA: {score_anomalo:.4f}")

plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.title(f"Controllo Qualità: PASSATO\nAnomaly Score: {score_sano:.4f}", color='green', fontsize=14, fontweight='bold')
plt.imshow(img_sana, cmap='gray')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title(f"Controllo Qualità: FALLITO (Anomalia)\nAnomaly Score: {score_anomalo:.4f}", color='red', fontsize=14, fontweight='bold')
plt.imshow(img_anomala, cmap='gray')
plt.axis('off')

plt.tight_layout()
plt.show()

# risultati

## confronto

In [ ]:
import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import roc_curve, auc
from skimage.feature import local_binary_pattern
from skimage.filters.rank import entropy
from skimage.morphology import disk
from tqdm import tqdm

train_dir = 'data/train/good'
tutte_sane = sorted([os.path.join(train_dir, f) for f in os.listdir(train_dir)])
test_sane_paths = tutte_sane[-50:] 

anom_dir = 'data/test/anomaly'
test_anom_paths = sorted([os.path.join(anom_dir, f) for f in os.listdir(anom_dir)])[:50]

test_paths = test_sane_paths + test_anom_paths
y_true = [0] * len(test_sane_paths) + [1] * len(test_anom_paths)

print(f"Totale immagini in test: {len(test_paths)} (50 Sane, 50 Anomale)")


def baseline_score(img_path):
    img = cv2.imread(img_path, 0)
    radius = 3
    n_points = 8 * radius
    
    lbp_img = local_binary_pattern(img, n_points, radius, method='uniform')
    return np.std(lbp_img)

def moco_score(img_path):
    img = Image.open(img_path).convert('RGB') 
    img_t = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        feat = encoder(img_t)
        feat = torch.nn.functional.normalize(feat, dim=1)

    similarities = torch.mm(feat, normal_bank.T)
    return 1.0 - torch.max(similarities).item()

scores_baseline = []
scores_moco = []

print("\n Valutazione in corso...")
for path in tqdm(test_paths):
    scores_baseline.append(baseline_score(path))
    scores_moco.append(moco_score(path))

fpr_b, tpr_b, _ = roc_curve(y_true, scores_baseline)
roc_auc_baseline = auc(fpr_b, tpr_b)

fpr_m, tpr_m, _ = roc_curve(y_true, scores_moco)
roc_auc_moco = auc(fpr_m, tpr_m)

plt.figure(figsize=(10, 8))
plt.plot(fpr_b, tpr_b, color='darkorange', lw=2, linestyle='--', label=f'Pipeline 1: Baseline LBP (AUC = {roc_auc_baseline:.2f})')
plt.plot(fpr_m, tpr_m, color='blue', lw=2, label=f'Pipeline 2: MoCo ResNet18 (AUC = {roc_auc_moco:.2f})')
plt.plot([0, 1], [0, 1], color='gray', lw=1, linestyle=':', label='Tiro a caso (AUC = 0.50)')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Tasso di Falsi Positivi (Sane scambiate per anomale)', fontsize=12)
plt.ylabel('Tasso di Veri Positivi (Anomalie trovate)', fontsize=12)
plt.title('Confronto Prestazioni: LBP Classico vs Contrastive Learning (MoCo)', fontsize=14, fontweight='bold')
plt.legend(loc="lower right", fontsize=12)
plt.grid(alpha=0.3)
plt.show()